In [31]:
import os
import json
import time
from typing import List
from groq import Groq


In [58]:
def save_json_atomic(path, data):
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

In [59]:
# prompt_sistema = f"""
    # Eres un analizador de texto estricto y un extractor de datos de alta precisión en español. 
    # Tu única tarea es identificar rangos de edad y categorizar actividades.\n\n

    # REGLAS INSTRUCCIONALES OBLIGATORIAS:\n
    # 1. Devuelve ÚNICAMENTE un objeto JSON plano con estos 4 campos: 
    # - 'app_id': (string) El identificador original recibido (ej. '50307956-jugar-arte-familia.json').
    # - 'es_aire_libre': (boolean o null) true si la actividad es en el exterior/aire libre, false si es en interiores/pabellones/teatros o si no se especifica o es ambiguo.
    # - 'edad_minima': (integer o null) edad mínima recomendada en años. Si dice 'de 1 a 5 años', es 1. 
    # - 'edad_maxima': (integer o null) edad máxima recomendada en años. Si dice 'de 1 a 5 años', es 5.
    # 3. Las claves 'edad_minima' y 'edad_maxima' deben ser números enteros (int). Si no se mencionan, su valor predeterminado es el string 'No mencionada'.\n
    # 4. LÓGICA DE RANGOS: Ante expresiones que indiquen un rango numérico de edad (estructuras del tipo 'de X a Y', 'entre X y Y', 'X-Y años'), estás obligado a evaluar de forma matemática ambos extremos: el número menor de la expresión SIEMPRE debe asignarse a 'edad_minima' y el número mayor SIEMPRE debe asignarse a 'edad_maxima'. No omitas ningún extremo bajo ninguna circunstancia, ignorando signos de exclamación o énfasis del texto.\n
    # 5. LÓGICA DE ADULTOS: Si el texto contiene los términos 'adulto', 'adultos' o 'mayor de edad', la 'edad_minima' debe establecerse automáticamente como el número entero 18.\n
    # 6. RESTRICCIÓN DE FORMATO: Está totalmente prohibido incluir introducciones, explicaciones, notas aclaratorias o bloques de código de marcado markdown (como ```json). Tu respuesta debe ser exclusivamente el JSON en texto crudo.

    # No incluyas texto de introducción, saludos ni explicaciones. Responde solo con el JSON.
                    
    # Actividades a procesar:
    # {bloque_input}
# """
                    

# prompt_sistema = f"""
#     Analiza la descripción de estas actividades del portal de datos de Madrid.
#     Devuelve OBLIGATORIAMENTE un único objeto JSON con una lista llamada 'actividades'.
#     Cada actividad dentro de la lista debe contener exactamente estos 4 campos:
#     - 'app_id': (string) El identificador original recibido (ej. '50307956-jugar-arte-familia.json').
#     - 'al_aire_libre': (boolean o null) true si la actividad es en el exterior/aire libre, false si es en interiores/pabellones/teatros, null si no se especifica o es ambiguo.
#     - 'edad_minima': (integer o null) edad mínima recomendada en años. Si dice 'de 1 a 5 años', es 1.
#     - 'edad_maxima': (integer o null) edad máxima recomendada en años. Si dice 'de 1 a 5 años', es 5.
    
#     No incluyas texto de introducción, saludos ni explicaciones. Responde solo con el JSON.

#     Actividades a procesar:
#     {bloque_input}
#     """

# prompt = f"""
#         Eres un analizador de texto estricto y un extractor de datos de alta precisión en español. 
#         Analiza la descripción de estas actividades del portal de datos de Madrid.

#         REGLAS INSTRUCCIONALES OBLIGATORIAS:
#         1. Devuelve OBLIGATORIAMENTE un único objeto JSON con una lista llamada 'actividades'.
#         2. Cada actividad dentro de la lista debe contener exactamente estos 4 campos:
#         - 'app_id': (string) El identificador original recibido (ej. '50307956-jugar-arte-familia.json').
#         - 'es_aire_libre': (boolean) true si la actividad es en el exterior/aire libre, false si es en interiores/pabellones/teatros o si no se especifica o es ambiguo.
#         - 'edad_minima': (integer o null) edad mínima recomendada en años. Si dice 'de 1 a 5 años', es 1. 
#         - 'edad_maxima': (integer o null) edad máxima recomendada en años. Si dice 'de 1 a 5 años', es 5.
#         3. LÓGICA DE RANGOS: Ante expresiones que indiquen un rango numérico de edad (estructuras del tipo 'de X a Y', 'entre X y Y', 'X-Y años'), estás obligado a evaluar de forma matemática ambos extremos: el número menor de la expresión SIEMPRE debe asignarse a 'edad_minima' y el número mayor SIEMPRE debe asignarse a 'edad_maxima'. No omitas ningún extremo bajo ninguna circunstancia, ignorando signos de exclamación o énfasis del texto.
#         4. LÓGICA DE ADULTOS: Si el texto contiene los términos 'adulto', 'adultos' o 'mayor de edad', la 'edad_minima' debe establecerse automáticamente como el número entero 18.

#         No incluyas texto de introducción, saludos ni explicaciones. Responde solo con el JSON.
                        
#         Actividades a procesar:
#         {bloque_input}
#     """

In [60]:
# 1. Inicializamos el cliente de Groq utilizando el secreto de GitHub
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def clasificar_lote_groq(lote_textos: List[dict]) -> List[dict]:
    bloque_input = ""
    for item in lote_textos:
        # Usamos tus campos exactos del JSON de Madrid
        bloque_input += f"--- APP_ID: {item['app_id']} ---\n{item['description']}\n--- FIN ---\n\n"

    prompt = f"""
Eres un sistema experto en extracción de datos (NER) y procesamiento de lenguaje natural de alta precisión. Tu tarea es analizar descripciones de actividades culturales y de ocio del Ayuntamiento de Madrid para estructurarlas en un formato JSON estricto.

### INSTRUCCIONES DE PROCESAMIENTO
1. Analiza detenidamente el identificador del archivo (`app_id`) y el texto de la descripción proporcionada.
2. Evalúa las condiciones ambientales para determinar si la actividad ocurre al aire libre.
3. Extrae los rangos de edad aplicando las reglas lógicas e infiriendo los límites numéricos según el contexto de forma matemática.

### LÓGICA DE EXTRACCIÓN DE EDADES (CRÍTICA)
- Expresiones de rango ("de X a Y", "entre X y Y", "X-Y años"): 'edad_minima' = X, 'edad_maxima' = Y.
- Expresiones de límite inferior ("a partir de X años", "mayores de X años", ">X"): 'edad_minima' = X, 'edad_maxima' = null.
- Expresiones de límite superior ("hasta X años", "menores de X años", "<X"): 'edad_minima' = null, 'edad_maxima' = X.
- Palabras clave de infancia: Si menciona "bebés" sin especificar edad, asume 'edad_minima' = 0.
- Palabras clave de adultos: Si menciona "adultos", "público adulto" o "mayores de edad", asume 'edad_minima' = 18.
- Sin restricciones ("para todos los públicos", "público familiar" sin rangos): 'edad_minima' = 0, 'edad_maxima' = null.
- En caso de total ambigüedad o ausencia de datos: Ambos campos deben ser null.

### FORMATO DE SALIDA REQUERIDO
Devuelve EXCLUSIVAMENTE un objeto JSON válido que cumpla estrictamente con el siguiente esquema de Typescript (no incluyes bloques de código markdown, ni texto introductorio, ni notas aclaratorias):

{{
  "actividades": [
    {{
      "app_id": string,       // Nombre del archivo original completo (ej: "50307956-jugar.json")
      "es_aire_libre": boolean, // true SOLO si se especifica explícitamente que la actividad es en exterior/aire libre/parque. false si es interior (teatros, museos, centros culturales), ambiguo o no especificado.
      "edad_minima": number | null, // Número entero en años.
      "edad_maxima": number | null  // Número entero en años.
    }}
  ]
}}

### DATOS A PROCESAR:
{bloque_input}
"""

    # prompt = f"""
    # Analiza la descripción de estas actividades del portal de datos de Madrid.
    # Devuelve OBLIGATORIAMENTE un único objeto JSON con una lista llamada 'actividades'.
    # Cada actividad dentro de la lista debe contener exactamente estos 4 campos:
    # - 'app_id': (string) El identificador original recibido (ej. '50307956-jugar-arte-familia.json').
    # - 'al_aire_libre': (boolean o null) true si la actividad es en el exterior/aire libre, false si es en interiores/pabellones/teatros, null si no se especifica o es ambiguo.
    # - 'edad_minima': (integer o null) edad mínima recomendada en años. Si dice 'de 1 a 5 años', es 1.
    # - 'edad_maxima': (integer o null) edad máxima recomendada en años. Si dice 'de 1 a 5 años', es 5.
    
    # No incluyas texto de introducción, saludos ni explicaciones. Responde solo con el JSON.

    # Actividades a procesar:
    # {bloque_input}
    # """

    for intento in range(3):
        try:
            completion = client.chat.completions.create(
                # model="llama3-8b-8192", # Modelo ultrasónico y gratuito
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,        # Temperatura 0 para evitar inventos y ser ultra preciso
                response_format={"type": "json_object"}  # Forzamos a Groq a devolver JSON estricto
            )
            
            # Parseamos el texto devuelto a un diccionario de Python
            resultado_json = json.loads(completion.choices[0].message.content)
            return resultado_json.get("actividades", [])
            
        except Exception as e:
            print(f"Error en intento {intento+1}: {e}")
            if intento < 2:
                print(f"Aviso: Intento {intento+1} falló. Esperando 10 segundos antes de reintentar...")
                time.sleep(10)
            else:
                raise e


In [ ]:
# --- FLUJO PRINCIPAL ---
# Nombres de tus archivos
archivo_entrada = "../data/actividades_procesadas.json"
archivo_salida = "../data/resultados_clasificados.json"

# 1. Cargar el histórico existente (lo procesado en días anteriores)
historico_clasificado = {}
if os.path.exists(archivo_salida):
    with open(archivo_salida, mode="r", encoding="utf-8") as f:
        try:
            datos_viejos = json.load(f)
            # Lo convertimos a diccionario indexado por app_id para buscar rápido
            historico_clasificado = {item["app_id"]: item for item in datos_viejos}
            print(f"Cargadas {len(historico_clasificado)} actividades ya clasificadas del histórico.")
        except json.JSONDecodeError:
            print("El archivo de salida estaba vacío o corrupto. Se creará de nuevo.")

# 2. Cargar el catálogo del día actual y FILTRAR los nuevos
nuevas_actividades = []
if os.path.exists(archivo_entrada):
    with open(archivo_entrada, mode="r", encoding="utf-8") as f:
        catalogo_actual = json.load(f)
        
        if isinstance(catalogo_actual, list):
            for item in catalogo_actual:
                uid = item.get("app_id")
                desc = item.get("description", "")
                if desc == "":
                    continue  # Saltamos actividades sin descripción
                else:
                    # 🎯 LA MAGIA: Si NO está en el histórico, es nuevo. Lo guardamos para procesar.
                    if uid not in historico_clasificado:
                        nuevas_actividades.append({
                            "app_id": uid,
                            "description": desc
                        })
    print(f"Total actividades hoy: {len(catalogo_actual)}. -> ¡Nuevas a procesar: {len(nuevas_actividades)}!")
else:
    print(f"No se encontró el archivo {archivo_entrada}.")
    exit(0)

# 3. Procesar SOLO los nuevos (si los hay)
if nuevas_actividades:
    tamanio_lote = 2
    total_lotes = -(-len(nuevas_actividades) // tamanio_lote)
    
    for i in range(0, len(nuevas_actividades), tamanio_lote):
        lote = nuevas_actividades[i:i+tamanio_lote]
        print(f"Procesando lote nuevo {i // tamanio_lote + 1} de {total_lotes}...")
        
        resultados_lote = clasificar_lote_groq(lote)
        
        # Integrar los nuevos resultados en nuestro diccionario histórico
        for res in resultados_lote:
            historico_clasificado[res["app_id"]] = res
        
        time.sleep(12)
        
        # 4. Guardar la lista total actualizada (viejos + nuevos)
        # with open(archivo_salida, mode="w", encoding="utf-8") as f:
        #     json.dump(list(historico_clasificado.values()), f, indent=2, ensure_ascii=False)
        # Guardar incrementalmente para evitar perder trabajo
        try:
            save_json_atomic(archivo_salida, historico_clasificado)
        except Exception as e:
            print(f"No se pudo guardar {archivo_salida}: {e}")

    print(f"¡Hecho! El histórico se ha actualizado y ahora tiene {len(historico_clasificado)} registros.")
else:
    print("Genial. No hay actividades nuevas hoy. Nada que enviar a Groq.")

Total actividades hoy: 1007. -> ¡Nuevas a procesar: 663!
Procesando lote nuevo 1 de 332...
Procesando lote nuevo 2 de 332...
Procesando lote nuevo 3 de 332...
Procesando lote nuevo 4 de 332...
Procesando lote nuevo 5 de 332...
Procesando lote nuevo 6 de 332...
Procesando lote nuevo 7 de 332...
Procesando lote nuevo 8 de 332...


In [ ]:
historico_clasificado